# Weighted Linear Least Squares

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/themintlab/ExecutableEngineering/blob/main/chapters/interpolation_and_curve_fitting/curve_fitting/weighted_linear_least_squares.ipynb)


In [ ]:
import numpy as np
import plotly.graph_objects as go

try:
    import executable_engineering as exe
except ImportError:
    %pip install -q executable_engineering
    import executable_engineering as exe


## The Problem with Unweighted Residuals

As with most approximate methods, the moment we start to move away from an *exact solution* subtle effects start to show up! 

Notice we are minimizing the *residuals*, but there is a subtle problem with the standard least squares formulation. Consider the inconsistent system from the pseudoinverse section:

$$\begin{aligned}
2 c + t &= 15  \\
c - t  &= 10 \\
c + 2 t &= 14
\end{aligned} $$

If we were to multiply the second equation by 10, the mathematical line it represents remains geometrically identical. However, the *residuals* of the second equation would scale up by an order of magnitude:

$$\begin{aligned}
2 c + t &= 15  \\
10 c - 10 t  &= 100 \\
c + 2 t &= 14
\end{aligned} $$

Because the residuals for the second equation are now 10 times larger for the same geometric distance, the least squares algorithm will prioritize minimizing the error on the second equation, dragging our 'center of mass' solution towards it!

What does this remind you of?

## Introducing Weights

Jacobi (diagonal) preconditioning!

$$P^{-1} = \begin{bmatrix} W_{1} & 0 & 0 \\ 0 & W_{2} & 0 \\ 0 & 0 & W_{3} \end{bmatrix}$$

If we define $r_1, r_2, r_3$:

$$\begin{aligned}
2 c + t - 15 &= r_1 \\
c - t - 10 &= r_2 \\
c + 2 t - 14 &= r_3
\end{aligned} $$

We say the residuals can be *weighted*, i.e.: the least squares problem becomes,

$$ \text{Min}_{\mathbf{x}} \ \sum W_i^2 r_i^2 $$

Let's code it and see how applying this diagonal matrix shifts the least squares solution!

In [ ]:
A = np.array([[2, 1], [1, -1], [1, 2]])
b = np.array([15, 10, 14])

#~~ Question: What's the preconditioner? How do we apply it?
Pi = np.diag([1, 10, 1])
print("Preconditioner / Weights Matrix:\n", Pi)
A_weighted = Pi @ A
b_weighted = Pi @ b
###

x_lsq, _, _, _ = np.linalg.lstsq(A_weighted, b_weighted, rcond=None)
print("\nWeighted Least Squares Solution (c, t):", x_lsq)

x_unweighted, _, _, _ = np.linalg.lstsq(A, b, rcond=None)
print("Unweighted Least Squares Solution (c, t):", x_unweighted)

# Prepare data for plotting
x_line = np.linspace(0, 10, 100)
y1 = 15 - 2*x_line
y2 = x_line - 10
y3 = (14 - x_line) / 2

# Plot the lines and the solutions
fig = go.Figure()
fig.add_trace(go.Scatter(x=x_line, y=y1, mode='lines', name='2c + t = 15'))
fig.add_trace(go.Scatter(x=x_line, y=y2, mode='lines', name='c - t = 10'))
fig.add_trace(go.Scatter(x=x_line, y=y3, mode='lines', name='c + 2t = 14'))

fig.add_trace(go.Scatter(x=[x_unweighted[0]], y=[x_unweighted[1]], mode='markers', 
                         marker=dict(size=10, color='black'), 
                         name='Unweighted Solution'))

fig.add_trace(go.Scatter(x=[x_lsq[0]], y=[x_lsq[1]], mode='markers', 
                         marker=dict(size=14, color='red', symbol='star'), 
                         name='Weighted Solution (Equation 2 weighted 10x)'))

fig.update_layout(title='Weighted vs Unweighted Least Squares for Inconsistent System', 
                  xaxis_title='c', yaxis_title='t',
                  yaxis=dict(range=[-10, 10]))
fig.show()


## Measurement Uncertainty as Weights

Weights are an excellent way to introduce measurement uncertainty into your fit!

If you have multiple measurements, $y_i^j$, at a single point, $x_i$, then you can find the mean,

$$
\tilde{y}_i = \text{mean}_j (y_i^j)   
$$

and its variance,

$$
\sigma_i^2 = \text{mean}\left([y_i^j - \tilde{y}_i]^2 \right)
$$ 

and weight by the inverse of the variance, $W_i = \frac{1}{\sigma_i^2}$.

Most package implementations of weighted least squares regression have an option to input the weights, but exact usage varies. Consult the documentation to determine how to use it in practice.